<a href="https://colab.research.google.com/github/AR-Ashik-9997/Phitron-practice-problem/blob/main/CNN_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset,DataLoader,Subset
from PIL import Image
import kagglehub

In [2]:
torch.manual_seed(42)
device=torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(f"using Device {device}")

using Device cuda


In [3]:
# Download Dataset
path=kagglehub.dataset_download("mohitsingh1804/plantvillage")
print("path",path)

Using Colab cache for faster access to the 'plantvillage' dataset.
path /kaggle/input/plantvillage


In [4]:
TRAIN_PATH=os.path.join(path,"PlantVillage","train")
VAL_PATH=os.path.join(path,"PlantVillage","val")

In [5]:
transforms=transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3,std=[0.5]*3)
])

In [17]:
abc=sorted(os.listdir(TRAIN_PATH))
abc

['Apple___Apple_scab',
 'Apple___Black_rot',
 'Apple___Cedar_apple_rust',
 'Apple___healthy',
 'Blueberry___healthy',
 'Cherry_(including_sour)___Powdery_mildew',
 'Cherry_(including_sour)___healthy',
 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
 'Corn_(maize)___Common_rust_',
 'Corn_(maize)___Northern_Leaf_Blight',
 'Corn_(maize)___healthy',
 'Grape___Black_rot',
 'Grape___Esca_(Black_Measles)',
 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
 'Grape___healthy',
 'Orange___Haunglongbing_(Citrus_greening)',
 'Peach___Bacterial_spot',
 'Peach___healthy',
 'Pepper,_bell___Bacterial_spot',
 'Pepper,_bell___healthy',
 'Potato___Early_blight',
 'Potato___Late_blight',
 'Potato___healthy',
 'Raspberry___healthy',
 'Soybean___healthy',
 'Squash___Powdery_mildew',
 'Strawberry___Leaf_scorch',
 'Strawberry___healthy',
 'Tomato___Bacterial_spot',
 'Tomato___Early_blight',
 'Tomato___Late_blight',
 'Tomato___Leaf_Mold',
 'Tomato___Septoria_leaf_spot',
 'Tomato___Spider_mites Two-spotted_

In [32]:
smaples=[]
b={cls_name:idx for idx,cls_name in enumerate(abc)}
print(b)
for c in abc:
  d=os.path.join(TRAIN_PATH,c)
  print(d)

  for img in os.listdir(d):
    img_path=os.path.join(d,img)
    # print(img_path)

    if os.path.isfile(img_path):
      label=b[c]
      smaples.append((img_path,label))
  # smaples

{'Apple___Apple_scab': 0, 'Apple___Black_rot': 1, 'Apple___Cedar_apple_rust': 2, 'Apple___healthy': 3, 'Blueberry___healthy': 4, 'Cherry_(including_sour)___Powdery_mildew': 5, 'Cherry_(including_sour)___healthy': 6, 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 7, 'Corn_(maize)___Common_rust_': 8, 'Corn_(maize)___Northern_Leaf_Blight': 9, 'Corn_(maize)___healthy': 10, 'Grape___Black_rot': 11, 'Grape___Esca_(Black_Measles)': 12, 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 13, 'Grape___healthy': 14, 'Orange___Haunglongbing_(Citrus_greening)': 15, 'Peach___Bacterial_spot': 16, 'Peach___healthy': 17, 'Pepper,_bell___Bacterial_spot': 18, 'Pepper,_bell___healthy': 19, 'Potato___Early_blight': 20, 'Potato___Late_blight': 21, 'Potato___healthy': 22, 'Raspberry___healthy': 23, 'Soybean___healthy': 24, 'Squash___Powdery_mildew': 25, 'Strawberry___Leaf_scorch': 26, 'Strawberry___healthy': 27, 'Tomato___Bacterial_spot': 28, 'Tomato___Early_blight': 29, 'Tomato___Late_blight': 30, 'Tomato

In [28]:
class MultiClassClassification(Dataset):
  def __init__(self,root_dir,transforms=None):
    super().__init__()

    self.samples=[]
    self.transforms=transforms
    self.classes=sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,d))])
    self.class_to_idx={cls_name: idx for idx,cls_name in enumerate(self.classes)}

    self.samples = [
        (os.path.join(root_dir, c, image_path), self.class_to_idx[c])
        for c in self.classes
        for image_path in os.listdir(os.path.join(root_dir, c))
        if os.path.isfile(os.path.join(root_dir, c, image_path))
        ]

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    img_path,label=self.samples[idx]
    image=Image.open(img_path).convert("RGB")

    if self.transforms:
      image=self.transforms(image)
    return image,label

In [29]:
train_dataset_full=MultiClassClassification(TRAIN_PATH,transforms)
test_dataset_full=MultiClassClassification(VAL_PATH,transforms)
num_classes=len(train_dataset_full.classes)

print(f"Num_class:{num_classes}")
print(f"full_train_size:{len(train_dataset_full)}")
print(f"full_Test_size:{len(test_dataset_full)}")

Num_class:38
full_train_size:43444
full_Test_size:10861


In [ ]:
train_dataset=Subset(train_dataset_full,list(range(100,len(train_dataset_full))))
test_dataset=Subset(test_dataset_full,list(range(100,len(test_dataset_full))))

In [ ]:
pin=True if device.type=='cuda' else False
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=pin)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=True,pin_memory=pin)